## Move FHIR Bundles (Incremental via Auto Loader)
Uses Auto Loader (`cloudFiles` with `binaryFile` format) to incrementally read FHIR bundle files from a source volume, distribute writes to a destination volume via a Spark UDF, and track all moved files in a **streaming Delta table `file_tracker`**.

Re-running this notebook only processes **new files** that haven't been moved before (Auto Loader checkpoint handles deduplication).

**Parameters:**
- **catalog_use** – Target Unity Catalog catalog
- **schema_use** – Target schema within the catalog
- **volume_use** – Destination volume name
- **source_volume_path** – Full path to the source volume containing FHIR bundles

In [0]:
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
volume_use = dbutils.widgets.get("volume_use")
source_volume_path = dbutils.widgets.get("source_volume_path")

dest_volume_path = f"/Volumes/{catalog_use}/{schema_use}/{volume_use}"

print(f"Source: {source_volume_path}")
print(f"Destination: {dest_volume_path}")

In [0]:
import os
from pyspark.sql.functions import udf, col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, BooleanType, StringType

# Result struct: {file_moved: bool, new_path: str|null, message: str|null}
move_result_schema = StructType([
    StructField("file_moved", BooleanType(), False),
    StructField("new_path", StringType(), True),
    StructField("message", StringType(), True)
])

@udf(returnType=move_result_schema)
def move_file_udf(file_path: str, content: bytes, dest_base: str) -> dict:
    """Write file content to destination volume via FUSE mount, return move result."""
    try:
        filename = os.path.basename(file_path)
        new_path = os.path.join(dest_base, filename)
        if os.path.exists(new_path):
            return {"file_moved": False, "new_path": None, "message": "SKIPPED (exists)"}
        os.makedirs(dest_base, exist_ok=True)
        with open(new_path, "wb") as f:
            f.write(content)
        return {"file_moved": True, "new_path": new_path, "message": None}
    except Exception as e:
        return {"file_moved": False, "new_path": None, "message": str(e)}

In [0]:
checkpoint_path = f"/Volumes/{catalog_use}/{schema_use}/{volume_use}/_checkpoints/file_tracker"
table_name = f"{catalog_use}.{schema_use}.file_tracker"

# Auto Loader incrementally reads only NEW files via checkpoint tracking
df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .load(source_volume_path)
)

# Apply distributed move UDF, drop binary content, add audit timestamp
df_moved = (
    df_stream
    .withColumn("move_result", move_file_udf(col("path"), col("content"), lit(dest_volume_path)))
    .select(
        col("path").alias("source_path"),
        col("length").alias("file_size_bytes"),
        col("move_result.file_moved").alias("file_moved"),
        col("move_result.new_path").alias("new_path"),
        col("move_result.message").alias("message"),
        col("modificationTime").alias("source_modified_at"),
        current_timestamp().alias("moved_at")
    )
)

# Write to streaming Delta table — trigger(availableNow=True) processes all new files then stops
query = (
    df_moved.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()
print(f"Streaming write complete. Results written to: {table_name}")

In [0]:
# Show all tracked file moves across all runs
df_tracker = spark.table(f"{catalog_use}.{schema_use}.file_tracker")
print(f"Total files tracked: {df_tracker.count()}")
display(df_tracker.orderBy(col("moved_at").desc()))